In [ ]:
# Colab Cell 1 — Install dependencies
!pip install --upgrade pip >/dev/null
!pip install azure-storage-blob mysql-connector-python requests tqdm python-dotenv Pillow >/dev/null
!pip install openai


In [ ]:
# Colab Cell 2 — MySQL connection and table setup
import os
from datetime import datetime
import mysql.connector
from mysql.connector import errorcode

MYSQL_HOST = os.getenv("MYSQL_HOST", "cycleai.mysql.database.azure.com")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", "3306"))
MYSQL_USER = os.getenv("MYSQL_USER", "admindev")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD", "Zana81333")
MYSQL_DB = os.getenv("MYSQL_DB", "cycle_ai")

def get_mysql_conn():
    cfg = {
        "host": MYSQL_HOST,
        "port": MYSQL_PORT,
        "user": MYSQL_USER,
        "password": MYSQL_PASSWORD,
        "database": MYSQL_DB,
        "autocommit": False,
    }
    try:
        conn = mysql.connector.connect(**cfg)
        return conn
    except mysql.connector.Error as err:
        if err.errno == errorcode.ER_ACCESS_DENIED_ERROR:
            print("Something is wrong with your user name or password")
        elif err.errno == errorcode.ER_BAD_DB_ERROR:
            print("Database does not exist")
            raise

print("MySQL connected and table ensured: perception_poll_votes_ai")


In [ ]:
# Colab Cell 3 — Azure Blob connection and caching blob names
import os, io, random
from azure.storage.blob import BlobServiceClient
from PIL import Image

AZURE_CONN_STR = os.getenv("AZURE_CONN_STR", "")   # recommended: store in Colab secret or env
BLOB_CONTAINER = os.getenv("BLOB_CONTAINER", "london")

if not AZURE_CONN_STR:
    raise RuntimeError("AZURE_CONN_STR not set. Set as environment variable or inline in this cell.")

blob_svc = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
container_client = blob_svc.get_container_client(BLOB_CONTAINER)

print("Listing blobs in 'Images/' folder (this may take a few seconds)...")

IMAGE_PREFIX = "training/training/Images/"
all_image_blobs = container_client.list_blobs(name_starts_with=IMAGE_PREFIX)

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp')

blob_names = [
    blob.name for blob in all_image_blobs
    if blob.name.lower().endswith(IMAGE_EXTENSIONS)
]

print(f"Found {len(blob_names)} image files.")
if len(blob_names) < 2:
    raise RuntimeError(f"Need at least 2 blobs in container to sample pairs. Found: {len(blob_names)}")

def download_blob_to_pil(blob_name, timeout=30):
    """Download a blob and return it as a PIL Image."""
    blob_data = container_client.download_blob(blob_name, timeout=timeout).readall()
    img = Image.open(io.BytesIO(blob_data)).convert("RGB")
    return img

def sample_two_random_blobs():
    a, b = random.sample(blob_names, 2)
    return a, b

print("Azure blob helpers ready.")


In [ ]:
# Colab Cell: Check for Rapid/Spam Voting
import pandas as pd
import mysql.connector
import warnings

warnings.filterwarnings('ignore', category=UserWarning)

conn = get_mysql_conn()

sql_spam_check = """
SELECT
    t1.user_id,
    t1.voting_time AS vote_time_1,
    t2.voting_time AS vote_time_2,
    t1.image_name1,
    t1.image_name2,
    t1.vote AS vote_1,
    t2.vote AS vote_2,
    TIMESTAMPDIFF(MICROSECOND, t1.voting_time, t2.voting_time) / 1000000.0 AS time_diff_seconds
FROM perception_poll_votes_london t1
JOIN perception_poll_votes_london t2
    ON t1.user_id = t2.user_id
    AND t1.image_name1 = t2.image_name1
    AND t1.image_name2 = t2.image_name2
    AND t1.voting_time < t2.voting_time
    AND t2.voting_time < t1.voting_time + INTERVAL 1 SECOND;
"""

try:
    print("Checking for duplicate votes on the same pair...")
    df_spam = pd.read_sql(sql_spam_check, conn)

    if not df_spam.empty:
        print(f"\n⚠️ FOUND {len(df_spam)} DUPLICATE VOTES")
        print("These users voted on the same pair twice within 1 second:")

        pd.set_option('display.max_rows', 50)
        display(df_spam)

        print("\n--- Offenders Summary ---")
        summary = df_spam.groupby('user_id').size().reset_index(name='duplicate_count')
        display(summary.sort_values('duplicate_count', ascending=False))
    else:
        print("\n✅ Clean. No users found voting on the same pair multiple times within 1 second.")

except Exception as e:
    print(f"Error running check: {e}")
finally:
    conn.close()
    pass

In [ ]:
OPENAI_API_KEY = "YOUR_OPENAI_API_KEY"

In [ ]:
# Colab Cell 4 (Modified for GPT-4o)
import os, io, json, time, base64, hashlib, random, re
from tqdm import tqdm
from PIL import Image
import requests
import mysql.connector
from datetime import datetime
from time import sleep
import uuid
from IPython.display import display, HTML
import pandas as pd
import openai # <-- ADDED: Import OpenAI library

# ---------- CONFIG ----------
OPENAI_MODEL_ID = "gpt-4o" # <-- ADDED: Specify model
client = openai.OpenAI(api_key=OPENAI_API_KEY)

WORKER_ID = os.getenv("WORKER_ID", str(uuid.uuid4()))

N = 10
BATCH_COMMIT = 2
RATE_LIMIT_SECONDS = float(os.getenv("RATE_LIMIT_SECONDS", "0.1"))
RETRY_MAX = 5
SAMPLE_TIMEOUT = 30
DEFAULT_CONFIDENCE_ON_FAIL = 0.0

FULL_PROMPT_TEXT = (
    "You will see two images: Image A (first) and Image B (second).\n\n"
    "TASK: Decide which image looks SAFER for cyclists.\n\n"
    "Respond with ONE line only in the exact required format:\n"
    "[VOTE]: [Vote Number] - [long reason]\n\n"
    "Where Vote Number is:\n"
    "  0 - Both look equally safe\n"
    "  1 - Image A looks safer\n"
    "  2 - Image B looks safer\n\n"
    "Keep your reason long. DO NOT include any extra text.\n\n"
    "Examples (Use this format EXACTLY):\n"
    "[VOTE]: 1 - Image A has a dedicated bike lane.\n"
    "[VOTE]: 2 - Image B has fewer vehicles present.\n"
    "[VOTE]: 0 - Both show similar traffic conditions.\n"
    "\nNow decide which image looks safer."
)

vote_counts = {0: 0, 1: 0, 2: 0}
results = []

def pil_to_data_uri(pil_img, quality=85):
    b64 = pil_to_base64_jpeg(pil_img, quality=quality)
    return f"data:image/jpeg;base64,{b64}"

def display_pair(pil1, pil2, vote, reason, name1, name2, max_width=320):
    uri1 = pil_to_data_uri(pil1)
    uri2 = pil_to_data_uri(pil2)
    caption = f"<b>Vote:</b> {vote} &nbsp;&nbsp; <b>Reason:</b> {reason}<br><b>Files:</b> {name1} — {name2}"
    html = f"""
    <div style="display:flex; align-items:flex-start; gap:12px; margin-bottom:10px;">
      <div style="text-align:center;">
        <img src="{uri1}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>{name1}</small>
      </div>
      <div style="text-align:center;">
        <img src="{uri2}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>{name2}</small>
      </div>
      <div style="flex:1; padding-left:8px;">
        {caption}
      </div>
    </div>
    """
    display(HTML(html))


# ---------- UTILITIES ----------
def pil_to_base64_jpeg(pil_img, quality=85):
    buff = io.BytesIO()
    pil_img.save(buff, format="JPEG", quality=quality)
    return base64.b64encode(buff.getvalue()).decode("utf-8")

def idempotency_key(name1, name2):
    s = f"{name1}|{name2}"
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def upsert_vote(conn, rec):
    cur = conn.cursor()
    sql = """
    INSERT INTO perception_poll_votes_ai
      (image_name1, image_name2, vote, model_id, user_id, reason)
    VALUES (%s, %s, %s, %s, %s, %s)
    """
    params = (rec["image_name1"], rec["image_name2"], rec["vote"], rec["model_id"], rec["user_id"], rec.get("reason"))
    cur.execute(sql, params)
    cur.close()


# --- REPLACED `call_model_locally` with `call_model_openai` ---

def call_model_openai(prompt_text, pil1, pil2):
    """
    Calls the OpenAI GPT-4o API with two PIL images and returns the model's text response.
    """
    if not OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY is not set. Cannot call API.")

    try:
        uri1 = pil_to_data_uri(pil1)
        uri2 = pil_to_data_uri(pil2)

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Image A:"},
                    {"type": "image_url", "image_url": {"url": uri1}},
                    {"type": "text", "text": "\nImage B:"},
                    {"type": "image_url", "image_url": {"url": uri2}},
                    {"type": "text", "text": f"\n\n{prompt_text}"}
                ]
            }
        ]

        response = client.chat.completions.create(
            model=OPENAI_MODEL_ID,
            messages=messages,
            max_tokens=128,
            temperature=0
        )

        text_output = response.choices[0].message.content
        print(f"Raw model generated (OpenAI): {text_output!r}")
        return text_output

    except Exception as e:
        print(f"Error calling OpenAI API: {e}")
        raise

# ---------- MAIN LOOP ----------
print(f"Starting main loop: N={N}, OpenAI model={OPENAI_MODEL_ID}, worker={WORKER_ID}")
conn = get_mysql_conn()
insert_count = 0

try:
    blob_names
except NameError:
    from azure.storage.blob import BlobServiceClient
    IMAGE_PREFIX = "training/training/Images/"
    IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp')

    print(f"Connecting to Azure to list files starting with '{IMAGE_PREFIX}'...")

    blob_svc = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
    container_client = blob_svc.get_container_client(BLOB_CONTAINER)

    all_image_blobs = container_client.list_blobs(name_starts_with=IMAGE_PREFIX)

    blob_names = [
        blob.name for blob in all_image_blobs
        if blob.name.lower().endswith(IMAGE_EXTENSIONS)
    ]

    if not blob_names:
        raise RuntimeError(f"No image files found in the '{IMAGE_PREFIX}' directory of the container.")

    print(f"Found {len(blob_names)} image files to sample from.")

pbar = tqdm(range(N), desc="Model votes")
for i in pbar:
    name1, name2 = random.sample(blob_names, 2)

    pil1 = pil2 = None
    for attempt in range(RETRY_MAX):
        try:
            pil1 = download_blob_to_pil(name1)
            pil2 = download_blob_to_pil(name2)
            break
        except Exception as e:
            wait = 2 ** attempt
            print(f"Blob download error for pair ({name1},{name2}) attempt {attempt+1}: {e}; sleeping {wait}s")
            time.sleep(wait)
    if pil1 is None or pil2 is None:
        pbar.set_postfix(status="download_fail")
        continue

    try:
        raw = call_model_openai(FULL_PROMPT_TEXT, pil1, pil2)
    except Exception as e:
        print("Model call failed, skipping:", e)
        pbar.set_postfix(status="model_fail")
        continue

    txt = str(raw)

    print("-" * 50)
    print(f"Model Output for ({name1.split('/')[-1]}, {name2.split('/')[-1]}):")
    print(txt)
    print("-" * 50)

    match = re.search(r"^(?:\[VOTE\]\s*[:\-]\s*)?\s*([0-2])\s*[:\-]\s*(.*)$", txt.strip(), re.MULTILINE | re.IGNORECASE)
    if not match:
        print(f"No valid 0/1/2 vote parsed for pair ({name1}, {name2}). Raw model text (truncated): {txt[:200]}")
        continue

    vote_val = int(match.group(1))
    reason_text = match.group(2).strip()

    IMAGE_PREFIX = "training/training/Images/"
    clean_name1 = name1.replace(IMAGE_PREFIX, '', 1)
    clean_name2 = name2.replace(IMAGE_PREFIX, '', 1)

    rec = {
        "id": idempotency_key(clean_name1, clean_name2),
        "image_name1": clean_name1,
        "image_name2": clean_name2,
        "vote": int(vote_val),
        "model_id": OPENAI_MODEL_ID,
        "user_id": WORKER_ID,
        "reason": reason_text,
    }

    try:
        upsert_vote(conn, rec)
        insert_count += 1
    except Exception as e:
        print("DB insert failed for record:", e)
        conn.rollback()
        pbar.set_postfix(status="db_error")
        continue

    vote_counts[vote_val] = vote_counts.get(vote_val, 0) + 1
    results.append({
      "image1": clean_name1,
      "image2": clean_name2,
      "vote": vote_val,
      "reason": reason_text,
      "model_id": OPENAI_MODEL_ID,
      "user_id": WORKER_ID
    })

    try:
      display_pair(pil1, pil2, vote_val, reason_text, clean_name1.split('/')[-1], clean_name2.split('/')[-1])
    except Exception as e:
      print("Display failed:", e)

    if insert_count % BATCH_COMMIT == 0:
        try:
            conn.commit()
        except Exception as e:
            print("DB commit failed:", e)
            conn.rollback()

    pbar.set_postfix(last_vote=rec["vote"])

    time.sleep(RATE_LIMIT_SECONDS)

try:
    conn.commit()
except Exception:
    conn.rollback()
conn.close()
print("Done. Inserted/updated records:", insert_count)

if results:
    df = pd.DataFrame(results)
    display(df.head(50))
else:
    print("No successful results captured in 'results' list.")